
## FASE 0 — Extracción masiva de texto de PDFs con PyMuPDF
=======================================================

Procesa ~19k PDFs en paralelo usando ProcessPoolExecutor.

Salida: archivo JSONL (una línea por PDF) con la estructura:
{
  "id":             nombre del archivo (sin ruta),
  "source_folder":  carpeta de origen,
  "type":           null  ← se llenará en análisis posterior,
  "page_count":     número total de páginas,
  "pages":          [{"page": 0, "chars": 450, "text": "..."}, ...],
  "texto":          texto completo concatenado de todas las páginas,
  "error":          null | "mensaje de error"
}

Diseño:
  • ProcessPoolExecutor para paralelismo real (CPU-bound con fitz).
  • Checkpoint: al iniciar, lee el JSONL de salida y omite PDFs ya procesados.
  • Progreso: barra tqdm en tiempo real.
  • Errores: capturados por PDF, nunca detienen el proceso global.

Uso:
  python phase0_extract.py
  python phase0_extract.py --workers 8 --output resultados.jsonl


In [ ]:
# import fitz
# import json
# import os
# from concurrent.futures import ProcessPoolExecutor, as_completed
# from pathlib import Path
# from tqdm import tqdm

# # ---------------------------------------------------------------------------
# # Config
# # ---------------------------------------------------------------------------

# SOURCE_FOLDERS = ["7.pdfs_04", "7.pdfs_nan", "7.pdfs_resto"]
# OUTPUT_JSONL   = "fase0_resultados.jsonl"
# WORKERS        = os.cpu_count() or 4
# MIN_CHARS      = 10   # mínimo de chars para considerar que una página tiene texto real

# # ---------------------------------------------------------------------------
# # Worker (debe estar al nivel del módulo para ser pickleable)
# # ---------------------------------------------------------------------------

# def extract_pdf(args: tuple) -> dict:
#     pdf_path, folder_name = args
#     file_name = Path(pdf_path).name

#     record = {
#         "id":            file_name,
#         "source_folder": folder_name,
#         "type":          None,
#         "page_count":    0,
#         "pages":         [],
#         "texto":         "",
#         "error":         None,
#     }

#     try:
#         doc = fitz.open(pdf_path)
#         record["page_count"] = doc.page_count
#         full_text_parts = []

#         for page_num in range(doc.page_count):
#             try:
#                 text  = doc[page_num].get_text("text", sort=True).strip()
#                 chars = len(text)
#                 record["pages"].append({"page": page_num, "chars": chars, "text": text})
#                 if chars >= MIN_CHARS:
#                     full_text_parts.append(text)
#             except Exception as e:
#                 record["pages"].append({"page": page_num, "chars": 0, "text": "", "page_error": str(e)})

#         record["texto"] = "\n\n".join(full_text_parts)
#         doc.close()

#     except Exception as e:
#         record["error"] = str(e)

#     return record

# # ---------------------------------------------------------------------------
# # Recolectar PDFs
# # ---------------------------------------------------------------------------

# all_pdfs = []
# for folder in SOURCE_FOLDERS:
#     for pdf in Path(folder).rglob("*.pdf"):
#         all_pdfs.append((str(pdf), Path(folder).name))

# print(f"PDFs encontrados: {len(all_pdfs):,}")

# # ---------------------------------------------------------------------------
# # Checkpoint — leer IDs ya procesados
# # ---------------------------------------------------------------------------

# processed = set()
# if Path(OUTPUT_JSONL).exists():
#     with open(OUTPUT_JSONL, "r", encoding="utf-8") as f:
#         for line in f:
#             try:
#                 processed.add(json.loads(line.strip())["id"])
#             except Exception:
#                 pass

# pending = [(p, f) for p, f in all_pdfs if Path(p).name not in processed]
# print(f"Ya procesados: {len(processed):,} | Pendientes: {len(pending):,}")

# # ---------------------------------------------------------------------------
# # Extracción
# # ---------------------------------------------------------------------------

# with open(OUTPUT_JSONL, "a", encoding="utf-8") as out_f:
#     with ProcessPoolExecutor(max_workers=WORKERS) as executor:
#         futures = {executor.submit(extract_pdf, args): args for args in pending}

#         for future in tqdm(as_completed(futures), total=len(pending), unit="pdf"):
#             try:
#                 record = future.result(timeout=60)
#             except Exception as e:
#                 args   = futures[future]
#                 record = {
#                     "id": Path(args[0]).name, "source_folder": args[1],
#                     "type": None, "page_count": 0,
#                     "pages": [], "texto": "", "error": str(e),
#                 }

#             out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
#             out_f.flush()

PDFs encontrados: 19,115
Ya procesados: 0 | Pendientes: 19,115


  9%|▉         | 1809/19115 [00:15<02:05, 137.69pdf/s]

MuPDF error: library error: FT_New_Memory_Face(COIZJT+HiddenHorzOCR): unknown file format



 10%|▉         | 1837/19115 [00:16<02:15, 127.83pdf/s]

MuPDF error: library error: FT_New_Memory_Face(COIZJT+HiddenHorzOCR): unknown file format

MuPDF error: library error: FT_New_Memory_Face(COIZJT+HiddenHorzOCR): unknown file format



 64%|██████▍   | 12236/19115 [01:53<01:13, 93.82pdf/s] 

MuPDF error: syntax error: cannot find ExtGState resource 'GS7'



100%|██████████| 19115/19115 [03:03<00:00, 104.26pdf/s]


In [73]:
import pandas as pd
import re

df0 = pd.read_json("fase0_resultados.jsonl", lines=True)[["id", "source_folder", "page_count", "texto"]]


#### Estrategia es ir descartando conforme encontremos SOLICITUD DE REGISTRO en texto


In [136]:
import re

##====================================================
# Limpieza de texto
def limpiar_texto(x):
    if not isinstance(x, str):
        return x
    # Reemplazar símbolos raros por espacio
    texto = re.sub(r'[^\w\s]', ' ', x)
    # Colapsar múltiples espacios en uno solo
    texto = re.sub(r'\s+', ' ', texto)
    # Quitar espacios al inicio y final
    return texto.strip()

df = df0.copy()

df["texto"] = df["texto"].apply(limpiar_texto)

##====================================================

df['n_caracteres'] = df['texto'].astype(str).str.len()

print("-------------------------------------------------")
print(df['n_caracteres'].describe())

#contar filas con 0 caracteres
print("-------------------------------------------------")
print(df[df['n_caracteres'] == 0].shape[0])
##====================================================
## ID extraction
def extraer_id_y_suffix(texto):
    if not isinstance(texto, str):
        return pd.Series([None, None])
    clean = re.sub(r'\.pdf$', '', texto)
    match = re.search(r'_(\d+)$', clean)
    suffix = int(match.group(1)) if match else None
    clean_id = re.sub(r'_\d+$', '', clean)
    
    return pd.Series([clean_id, suffix])

df[['id_clean', 'suffix']] = df['id'].apply(extraer_id_y_suffix)
##====================================================



-------------------------------------------------
count     19115.000000
mean       9117.451844
std       10910.743174
min           0.000000
25%        6511.000000
50%        6691.000000
75%        7314.000000
max      467268.000000
Name: n_caracteres, dtype: float64
-------------------------------------------------
103


In [137]:
import re

TLDS  = r'(?:COM MX|GOB MX|EDU MX|ORG MX|NET MX|COM|MX|NET|ORG|EDU|MED)'
NOISE = r'(?:CONTIN|PÁGINA|DATOS|DOMICILIO|TELEF|NOMBRE|PRIMER|SEGUNDO|NACIONA)'

def extraer_email_zona(zona):
    z = zona.upper().strip()
    z = re.split(NOISE, z)[0].strip()
    z = re.sub(r'^(?:CORREO\s+)?\bELECTR\w+\s*', '', z).strip()  # quita "CORREO ELECTRÓNICO" al inicio
    if len(z) < 4:
        return None
    m = re.search(rf'((?:[A-Z0-9_.]+\s+){{1,4}}?)([A-Z0-9_.]+)\s+({TLDS})', z)
    if m:
        partes  = m.group(1).strip().split()
        dominio = m.group(2)
        tld     = m.group(3).replace(' ', '.')
        usuario = '.'.join(partes) if partes else dominio
        return f"{usuario}@{dominio}.{tld}"
    m2 = re.match(r'^([A-Z0-9_.]{3,})\s+([A-Z0-9_.]{3,})$', z)
    if m2:
        return f"{m2.group(1)}@{m2.group(2)}"   # sin TLD reconocido
    return None

def extraer_campos(texto):
    if not isinstance(texto, str):
        return {}
    t = texto

    def find_sec(patron):
        m = re.search(patron, t, re.IGNORECASE)
        return m.start() if m else -1

    idx_datos_gen = find_sec(r'Datos generales del o de los solicitante')
    idx_dom_sol   = find_sec(r'Domicilio del solicitante')
    idx_dom_notif = find_sec(r'Domicilio para o\w* y recibir')
    idx_signo     = find_sec(r'Datos del signo distintivo')

    fin_notif = idx_signo     if idx_signo     != -1 else len(t)
    fin_sol   = idx_dom_notif if idx_dom_notif != -1 else fin_notif

    email = None
    if idx_datos_gen != -1 and idx_dom_sol != -1:
        sec_datos = t[idx_datos_gen: idx_dom_sol]
        for m in re.finditer(r'Correo electr\w+', sec_datos, re.IGNORECASE):
            candidato = extraer_email_zona(sec_datos[m.end(): m.end() + 90])
            if candidato:
                email = candidato
                break

    mun_sol = ent_sol = None
    if idx_dom_sol != -1:
        sec = t[idx_dom_sol: fin_sol]
        m = re.search(
            r'Municipio o demarcaci\w+ territorial Localidad (.+?) Entidad federativa (.+?)'
            r'(?= Entre| P\xe1g| Calle| Dom| Correo| Contin)',
            sec, re.IGNORECASE)
        if m:
            mun_sol = m.group(1).strip().upper()
            ent_sol = m.group(2).strip().upper()

    mun_not = ent_not = None
    if idx_dom_notif != -1:
        sec = t[idx_dom_notif: fin_notif]
        m = re.search(
            r'Municipio o demarcaci\w+ territorial Localidad (.+?) Entidad federativa (.+?)'
            r'(?= Entre| P\xe1g| Calle| Dom| Correo| Contin)',
            sec, re.IGNORECASE)
        if m:
            mun_not = m.group(1).strip().upper()
            ent_not = m.group(2).strip().upper()

    return {
        'email_titular':            email,
        'municipio_solicitante':    mun_sol,
        'entidad_solicitante':      ent_sol,
        'municipio_notificaciones': mun_not,
        'entidad_notificaciones':   ent_not,
    }

campos = df["texto"].apply(extraer_campos)
df = pd.concat([df, pd.DataFrame(campos.tolist(), index=df.index)], axis=1)

In [138]:
# Flag
df['tiene_municipio_entidad'] = (
    df.groupby('id_clean')[['municipio_solicitante', 'entidad_solicitante']]
      .apply(lambda x: x.notna().all(axis=1).any())
      .reset_index(level=0, drop=True)
)

def seleccionar_mejor_fila(grupo):
    if len(grupo) == 1:
        return grupo.iloc[[0]]
    
    # === NUEVA LÓGICA SIMPLE QUE PEDISTE ===
    # Si el grupo tiene al menos un True → nos quedamos SOLO con las filas True
    if grupo['tiene_municipio_entidad'].any():          # <--- cambiado a .any()
        grupo_filtrado = grupo[grupo['tiene_municipio_entidad'] == True]
        
        if len(grupo_filtrado) > 0:                     # seguridad extra
            grupo = grupo_filtrado.copy()
    
    # Ahora aplicamos prioridades normales
    # 1. Email
    con_email = grupo[grupo['email_titular'].notna()]
    if len(con_email) > 0:
        return con_email.iloc[[0]]
    
    # 2. Municipio + Entidad de notificaciones
    con_notif = grupo[
        grupo['municipio_notificaciones'].notna() & 
        grupo['entidad_notificaciones'].notna()
    ]
    if len(con_notif) > 0:
        return con_notif.iloc[[0]]
    
    # 3. Último recurso: la primera fila
    return grupo.iloc[[0]]


# ====================== APLICAR ======================
df_final = (
    df.groupby('id_clean', group_keys=False)
      .apply(seleccionar_mejor_fila)
      .reset_index(drop=True)
)

# Recalcular el flag correctamente con la fila que quedó
df_final['tiene_municipio_entidad'] = (
    df_final['municipio_solicitante'].notna() & 
    df_final['entidad_solicitante'].notna()
)

print(f"Registros originales : {len(df):,}")
print(f"Registros finales    : {len(df_final):,}")

Registros originales : 19,115
Registros finales    : 17,773


/tmp/ipykernel_5972/2783322968.py:41: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(seleccionar_mejor_fila)


In [139]:
# Crear la lista de id_clean donde tiene_municipio_entidad es False
lista_id_sin_datos = df_final[df_final['tiene_municipio_entidad'] == False]['id_clean'].unique().tolist()

# Mostrar cuántos hay
print(f"Se encontraron {len(lista_id_sin_datos)} id_clean sin municipio/entidad")

# Guardar en archivo txt (uno por línea)
with open('2.8_id_clean_sin_municipio_entidad.txt', 'w', encoding='utf-8') as f:
    for id_clean in lista_id_sin_datos:
        f.write(id_clean + '\n')

print("Archivo guardado como: 2.8_id_clean_sin_municipio_entidad.txt")

df_final_1 = df_final[df_final['tiene_municipio_entidad'] == True]


Se encontraron 1553 id_clean sin municipio/entidad
Archivo guardado como: 2.8_id_clean_sin_municipio_entidad.txt


In [141]:
df_final_1.shape

(16220, 13)

In [145]:
df_final_2 = df_final_1[[
    "id_clean", 
    "email_titular", 
    "municipio_solicitante", 
    "entidad_solicitante", 
    "municipio_notificaciones", 
    "entidad_notificaciones"
]]

In [159]:
ESTADOS_NORM = {
    # CIUDAD DE MEXICO
    "CIUDAD DE MÉXICO": "CIUDAD DE MEXICO",
    "CDMX": "CIUDAD DE MEXICO",
    "DISTRITO FEDERAL": "CIUDAD DE MEXICO",
    "DF": "CIUDAD DE MEXICO",
    "CD DE MEXICO": "CIUDAD DE MEXICO",
    "CD MEXICO": "CIUDAD DE MEXICO",
    "CD MÉXICO": "CIUDAD DE MEXICO",
    "CUIDAD DE MEXICO": "CIUDAD DE MEXICO",
    "CUIDAD DE MÉXICO": "CIUDAD DE MEXICO",
    "CIUADAD DE MEXICO": "CIUDAD DE MEXICO",
    "CIUADAD DE MÉXICO": "CIUDAD DE MEXICO",
    "CIUDAD MÉXICO": "CIUDAD DE MEXICO",
    "CIUDAD E MEXICO": "CIUDAD DE MEXICO",
    "CIUDA DE MEXICO": "CIUDAD DE MEXICO",
    "CIUDADA DE MÉXICO": "CIUDAD DE MEXICO",
    "CIUDAD DE MÉX": "CIUDAD DE MEXICO",
    "CIIUDAD DE MEXICO": "CIUDAD DE MEXICO",
    "CIUFDAD DE MEXICO": "CIUDAD DE MEXICO",
    "CIDUAD DE MEXICO": "CIUDAD DE MEXICO",
    "CUIDADO DE MEXICO": "CIUDAD DE MEXICO",
    "CIIDAD DE MÉXICO": "CIUDAD DE MEXICO",
    "DISTRITO FEDETRAL": "CIUDAD DE MEXICO",
    # ESTADO DE MEXICO
    "ESTADO DE MÉXICO": "ESTADO DE MEXICO",
    "EDO DE MEXICO": "ESTADO DE MEXICO",
    "EDO DE MÉXICO": "ESTADO DE MEXICO",
    "EDO MEX": "ESTADO DE MEXICO",
    "EDO MEXICO": "ESTADO DE MEXICO",
    "EDO MÉXICO": "ESTADO DE MEXICO",
    "EDOMEX": "ESTADO DE MEXICO",
    "EDOMEX ZONA METROPOLITANA": "ESTADO DE MEXICO",
    "ESTADO MÉXICO": "ESTADO DE MEXICO",
    "ESTADO DE MEX": "ESTADO DE MEXICO",
    "ESTADO DE MEXIO": "ESTADO DE MEXICO",
    "ESTADO DE MEXIC": "ESTADO DE MEXICO",
    "ESTADO DE MEXCIO": "ESTADO DE MEXICO",
    "ESTADO DE MÉXCO": "ESTADO DE MEXICO",
    "ESTAD DE MEXICO": "ESTADO DE MEXICO",
    "ESTA DE MEXICO": "ESTADO DE MEXICO",
    "ESTADO MEXICO": "ESTADO DE MEXICO",
    "MÉXICO": "ESTADO DE MEXICO",
    "MEXICO": "ESTADO DE MEXICO",
    "MÉX": "ESTADO DE MEXICO",
    "DE MÉXICO": "ESTADO DE MEXICO",
    "DE MEXICO": "ESTADO DE MEXICO",
    "ESTADO": "ESTADO DE MEXICO",
    "ESTADO DE MEXIO": "ESTADO DE MEXICO",
    # JALISCO
    "JALISO": "JALISCO",
    "JALISC O": "JALISCO",
    "JAL": "JALISCO",
    # NUEVO LEON
    "NUEVO LEÓN": "NUEVO LEON",
    "NUEVO LEÒN": "NUEVO LEON",
    "NUEVO LEÑON": "NUEVO LEON",
    "NUEVOL LEON": "NUEVO LEON",
    "NL": "NUEVO LEON",
    # QUINTANA ROO
    "QUINTANA ROOEXUCI": "QUINTANA ROO",
    "QUINTANAROO": "QUINTANA ROO",
    "QUINTNA ROO": "QUINTANA ROO",
    "QUINTNA ROO": "QUINTANA ROO",
    "QUITANA ROO": "QUINTANA ROO",
    "QUITNANA ROO": "QUINTANA ROO",
    "QUINTA ROO": "QUINTANA ROO",
    "QUINTANARRO": "QUINTANA ROO",
    "Q ROO": "QUINTANA ROO",
    "Q R": "QUINTANA ROO",
    "CANCÚN QUINTANA ROO": "QUINTANA ROO",
    "CANCUN QUINTANA ROO": "QUINTANA ROO",
    # BAJA CALIFORNIA
    "BAJA CALIFORNIA NORTE": "BAJA CALIFORNIA",
    "BAJA CALIFRONIA SUR": "BAJA CALIFORNIA SUR",
    "BAJA CALIFORNIAME": "BAJA CALIFORNIA",
    "BAJA CALIFORNIAI SUR": "BAJA CALIFORNIA SUR",
    "BAJA CALIFORIA SUR": "BAJA CALIFORNIA SUR",
    "BAJA CALIFONRIA SUR": "BAJA CALIFORNIA SUR",
    "BAJA CALIFORNIA DEL SUR": "BAJA CALIFORNIA SUR",
    "BAJA CALIFORIA": "BAJA CALIFORNIA",
    "BAJA CALIOFORNIA": "BAJA CALIFORNIA",
    "MBAJA CALIFORNIA SUR": "BAJA CALIFORNIA SUR",
    "MBAJA CALIFORNIA": "BAJA CALIFORNIA",
    "BAJA CALIFORNNIA SUR": "BAJA CALIFORNIA SUR",
    "BANA CALIFORNIA": "BAJA CALIFORNIA",
    "CABO SAN LUCAS BAJA CALIFORNIA S": "BAJA CALIFORNIA SUR",
    "PENÍNSULA DE BAJA CALIFORNIA": "BAJA CALIFORNIA",
    "BAJA CALIFRONIA SUR": "BAJA CALIFORNIA SUR",
    "BC": "BAJA CALIFORNIA",
    "B C": "BAJA CALIFORNIA",
    "B C S": "BAJA CALIFORNIA SUR",
    "BCS": "BAJA CALIFORNIA SUR",
    # VERACRUZ
    "VERACRUZ DE IGNACIO DE LA LLAVE": "VERACRUZ",
    "VERACRUZ DE LA LLAVE": "VERACRUZ",
    "VERACRUZ IGNACIO DE LA LLAVE": "VERACRUZ",
    "VERACRÚZ": "VERACRUZ",
    "VERCARUZ": "VERACRUZ",
    "VERACRUZ IGNACIO DE LA LLAVE": "VERACRUZ",
    "VERACCRUZ": "VERACRUZ",
    # MICHOACAN
    "MICHOACÁN": "MICHOACAN",
    "MICHOACÁN DE OCAMPO": "MICHOACAN",
    "MICHOACAN DE OCAMPO": "MICHOACAN",
    "MICHAOCAN": "MICHOACAN",
    "MICHOACAN DE OACAMPO": "MICHOACAN",
    "MICHOACAH": "MICHOACAN",
    # CHIHUAHUA
    "CHIHAUAHUA": "CHIHUAHUA",
    "CHIHAUHUA": "CHIHUAHUA",
    "CHIUAHUA": "CHIHUAHUA",
    "CHICHUAHUA": "CHIHUAHUA",
    "CHEHUAHUA": "CHIHUAHUA",
    "CHIHUHUA": "CHIHUAHUA",
    # QUERETARO
    "QUERÉTARO": "QUERETARO",
    "QURÉTARO": "QUERETARO",
    # YUCATAN
    "YUCATÁN": "YUCATAN",
    "YUCTAN": "YUCATAN",
    "YUCAAN": "YUCATAN",
    "YUCATAN YUC": "YUCATAN",
    # OAXACA
    "OXACA": "OAXACA",
    # CHIAPAS
    "CHIAPASS": "CHIAPAS",
    "CHIPAS": "CHIAPAS",
    "CHIS": "CHIAPAS",
    # SAN LUIS POTOSI
    "SAN LUIS POTOSÍ": "SAN LUIS POTOSI",
    "SAN LUÍS POTOSÍ": "SAN LUIS POTOSI",
    "SAN LUIS POSTOSI": "SAN LUIS POTOSI",
    # COAHUILA
    "COAHUILA DE ZARAGOZA": "COAHUILA",
    "COHUILA": "COAHUILA",
    "CUAHUILA": "COAHUILA",
    "COAHILA": "COAHUILA",
    "COAH": "COAHUILA",
    # GUANAJUATO
    "GUANAJUANTO": "GUANAJUATO",
    "GUANANJUATO": "GUANAJUATO",
    "GTO": "GUANAJUATO",
    # DURANGO
    "DUANGO": "DURANGO",
    # GUERRERO
    "GUERERO": "GUERRERO",
    "GUERREO": "GUERRERO",
    "GERRERO": "GUERRERO",
    "ESTADO DE GUERRERO": "GUERRERO",
    # SONORA
    "ESTADO DE SONORA": "SONORA",
    # TAMAULIPAS
    "TAMAULIMAS": "TAMAULIPAS",
    "TAMULIPAS": "TAMAULIPAS",
    # NAYARIT
    "NAY": "NAYARIT",
    # PUEBLA
    "ESTADO DE PUEBLA": "PUEBLA",
    "PUEBLA PUE": "PUEBLA",
    "PUEBLA HUAUCHINANGO": "PUEBLA",
    # HIDALGO
    "ESTADO DE HIDALGO": "HIDALGO",
    # AGUASCALIENTES
    "AGUASCALEINTES": "AGUASCALIENTES",
    "AGUASCALIENES": "AGUASCALIENTES",
    # SINALOA
    "SIN": "SINALOA",
    # ciudades que son el municipio no el estado — mejor dejar como x
    # (MERIDA=Yucatan, MONTERREY=NL, GUADALAJARA=Jalisco, etc. pero sin contexto
    #  extra son ambiguos: los mapeamos al estado correspondiente)
    "MÉRIDA": "YUCATAN",
    "MERIDA": "YUCATAN",
    "MERIDA YUCATAN": "YUCATAN",
    "CANCUN": "QUINTANA ROO",
    "MONTERREY": "NUEVO LEON",
    "GUADALAJARA": "JALISCO",
    "IZTAPALAPA": "CIUDAD DE MEXICO",
    "NAUCALPAN": "ESTADO DE MEXICO",
    "NAUCALPAN DE JUAREZ": "ESTADO DE MEXICO",
    "TLALNEPANTLA DE BAZ": "ESTADO DE MEXICO",
    "HUIXQUILUCAN": "ESTADO DE MEXICO",
    "TOLUCA": "ESTADO DE MEXICO",
    "VALLE DE CHALCO": "ESTADO DE MEXICO",
    "MORELIA": "MICHOACAN",
    "HERMOSILLO": "SONORA",
    "IRAPUATO": "GUANAJUATO",
    "CELAYA": "GUANAJUATO",
    "XALAPA": "VERACRUZ",
    "VILLAHERMOSA": "TABASCO",
    "SALTILLO": "COAHUILA",
    "NUEVO LAREDO": "TAMAULIPAS",
    "ENSENADA": "BAJA CALIFORNIA",
    "TIJUANA": "BAJA CALIFORNIA",
    "SANTA CATARINA NUEVO LEON": "NUEVO LEON",
    "SOLEDAD DE GRACIANO SÁNCHEZ": "SAN LUIS POTOSI",
    "EDOMEX ZONA METROPOLITANA": "ESTADO DE MEXICO",
    "CIUDAD": "CIUDAD DE MEXICO",
}

# Valores que definitivamente NO son estados de México
NO_MEXICO = {
    "FLORIDA", "INDIANA", "NEW JERSEY", "CALIFORNIA", "TEXAS", "PENSILVANIA",
    "NUEVA YORK", "SP", "BOGOTÁ", "MADRID", "CHILE", "PANAMA", "BENOS AIRES",
    "HUBEI", "GUANGDONG",
    # dudosos / sin sentido
    "SOLTERO A", "CASADO A", "PLEASE SELECT",
    "NUEVO", "SAN PEDRO HUAQUILPAN",
    "YUCATAN YUC", "ESTADO", "CIUDAD",
}

ESTADOS_VALIDOS = {
    "AGUASCALIENTES", "BAJA CALIFORNIA", "BAJA CALIFORNIA SUR", "CAMPECHE",
    "CHIAPAS", "CHIHUAHUA", "CIUDAD DE MEXICO", "COAHUILA", "COLIMA",
    "DURANGO", "ESTADO DE MEXICO", "GUANAJUATO", "GUERRERO", "HIDALGO",
    "JALISCO", "MICHOACAN", "MORELOS", "NAYARIT", "NUEVO LEON", "OAXACA",
    "PUEBLA", "QUERETARO", "QUINTANA ROO", "SAN LUIS POTOSI", "SINALOA",
    "SONORA", "TABASCO", "TAMAULIPAS", "TLAXCALA", "VERACRUZ", "YUCATAN",
    "ZACATECAS",
}

def normalizar_entidad(val):
    if not isinstance(val, str):
        return val
    v = val.strip().upper()
    return ESTADOS_NORM.get(v, v)

df_final_2["entidad_solicitante"] = df_final_2["entidad_solicitante"].apply(normalizar_entidad)

# Columna bd_x: "x" si el valor no es un estado válido de México
def marcar_no_mexico(val):
    if not isinstance(val, str):
        return None
    return "x" if val.strip().upper() not in ESTADOS_VALIDOS else None

df_final_2["bd_x"] = df_final_2["entidad_solicitante"].apply(marcar_no_mexico)

# Reporte
print(df_final_2["entidad_solicitante"].value_counts())
print(f"\nRegistros marcados como bd_x: {df_final_2['bd_x'].notna().sum()}")
print(df_final_2[df_final_2["bd_x"] == "x"]["entidad_solicitante"].value_counts())

entidad_solicitante
ESTADO DE MEXICO        1734
JALISCO                 1660
CIUDAD DE MEXICO        1240
NUEVO LEON              1186
YUCATAN                  989
QUINTANA ROO             986
SINALOA                  710
BAJA CALIFORNIA          707
GUANAJUATO               651
PUEBLA                   552
VERACRUZ                 500
CHIHUAHUA                495
OAXACA                   437
QUERETARO                415
MICHOACAN                407
SONORA                   358
BAJA CALIFORNIA SUR      333
HIDALGO                  292
COAHUILA                 290
TAMAULIPAS               276
CHIAPAS                  271
TABASCO                  250
AGUASCALIENTES           243
MORELOS                  193
SAN LUIS POTOSI          156
NAYARIT                  155
GUERRERO                 140
CAMPECHE                 137
COLIMA                   126
DURANGO                  110
ZACATECAS                102
TLAXCALA                  85
FLORIDA                    5
INDIANA                

/tmp/ipykernel_5972/467276365.py:228: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final_2["entidad_solicitante"] = df_final_2["entidad_solicitante"].apply(normalizar_entidad)
/tmp/ipykernel_5972/467276365.py:236: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final_2["bd_x"] = df_final_2["entidad_solicitante"].apply(marcar_no_mexico)


In [162]:
ESTADOS_NORM["CUAUTLANCINGO PUEBLA"] = "PUEBLA"
ESTADOS_NORM["EDO DE MEX"]           = "ESTADO DE MEXICO"
ESTADOS_NORM["CAMPECHE CAMPECHE"]    = "CAMPECHE"
ESTADOS_NORM["CIUAD DE MEXICO"]      = "CIUDAD DE MEXICO"
ESTADOS_NORM["BAJA CALIFORNA SUR"]   = "BAJA CALIFORNIA SUR"

# Re-aplicar
df_final_2["entidad_solicitante"] = df_final_2["entidad_solicitante"].apply(normalizar_entidad)
df_final_2["bd_x"] = df_final_2["entidad_solicitante"].apply(marcar_no_mexico)

/tmp/ipykernel_5972/2286812319.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final_2["entidad_solicitante"] = df_final_2["entidad_solicitante"].apply(normalizar_entidad)
/tmp/ipykernel_5972/2286812319.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final_2["bd_x"] = df_final_2["entidad_solicitante"].apply(marcar_no_mexico)


In [166]:
df_final_2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16220 entries, 1 to 17772
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   id_clean                  16220 non-null  object
 1   email_titular             15641 non-null  object
 2   municipio_solicitante     16220 non-null  object
 3   entidad_solicitante       16220 non-null  object
 4   municipio_notificaciones  16219 non-null  object
 5   entidad_notificaciones    16219 non-null  object
 6   bd_x                      29 non-null     object
dtypes: object(7)
memory usage: 1013.8+ KB


In [169]:
df_final_2['bd_x'] = df_final_2['bd_x'].fillna("pdf")


/tmp/ipykernel_5972/3539663332.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final_2['bd_x'] = df_final_2['bd_x'].fillna("pdf")


In [173]:
df_final_2 = df_final_2.rename(columns={
    'id_clean':              'id',
    'municipio_solicitante': 'municipality',
    'entidad_solicitante':   'state',
    'email_titular':         'titular_E-mail',
    'bd_x':                  'bd'
})

In [177]:
df_final_2.to_csv("8.data_pdf_scrapped.csv", index=False, encoding="utf-8-sig")